# Aliquot Sequences

In [1]:
# basic modules being imported
import numpy as np
import matplotlib.pyplot as plt
from math import *
from random import *

In [2]:
# -----   inefficient method for calculation   -----

def aliquot_basic(x, val_limit = 1e9, iter_limit = 30):
    '''
        Function that takes a positive integer x a creates an
        aliquot sequence.
        val_limit controls when the function stops if it goes
        above a certain number
        iter_limit automatically stops the function after a 
        set number of iterations
        Function returns a list of the aliquot sequence
    '''

    # creating an empty set for numbers to be added to
    aliquot_list = []

    # makes sure the function only does iterations up to the limit
    for i in range(iter_limit):

        # adds the latest number to the list and resets the value of x_divisors
        aliquot_list.append(x)
        x_divisors = 0

        # checks for each divisor of x and adds them together
        for j in range(1,x):
            if x % j == 0:
                x_divisors += j
        
        # replacing the old value with the new one
        x = x_divisors

        # stops iterations if it reaches 0
        if x == 0:
            aliquot_list.append(0)
            break
    
    return print(aliquot_list)



In [3]:
aliquot_basic(562)

[562, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284, 220, 284]


# A more efficient way to calculate the aliquot sequence

We've been able to generate the aliquot sequence for any given number, however our current model uses an inefficient method to do so. Going through each positive integer less than the target number and checking if they are a divisor involves lots of uneccessary tests, leading to long calculation times for large starting numbers and long sequences. Therefore, we looked into more efficient ways to calculate an aliquot number, in particular the part on multiplicative functions as mentioned in the Project Booklet.

### Multiplicative functions

A multiplicative function is a function such that, given two positive integers $a$ and $b$ are coprime $(gcd(a, b) = 1)$:

$f(ab) = f(a)f(b)$ and $f(1) = 1$

*insert proof that sum-of-divisors function is multiplicative*

Using the multiplicative property, we can separate any number we want into its distinct primes and calculate the sum of divisors separately, for example:

$$
\begin{align*}
\sigma(12) = &\sigma(2^2)\sigma(3)
\\
\sigma(12) = &(1 + 2 + 4)(1 + 3)
\\
\sigma(12) = &(7)(4) = 28
\end{align*}
$$

To make the aliquot product $s(n)$, we subtract the original number from the sum of divisors.

$$\begin{align*}  s(12) = 28 - 12 = 16    \end{align*}$$

This makes the process considerably quicker, as we can easily calculate the sum of divisors for powers of primes using the geometric sum formula. We can now define a formula for the sum of divisors for any positive integer:

For positive integer $n$ that can be expressed as a product of primes $p_1^{a_1}, p_2^{a_2}, ..., p_k^{a_k}$:

$$\begin{align*}  \sigma(n) = \prod_{i=1}^k\frac{p_i^{a_i+1}-1}{p_i-1}    \end{align*}$$

And subsequently:

$$\begin{align*}  s(n) = \sigma(n) - n    \end{align*}$$

### Applying the multiplicative function

In order to put this method into practise, we first have to find a way to split any given number into its prime factors. One way to do this could be to generate a list of primes up to a set limit, then define a function that uses the pregenerated list of primes to split the given number into its prime factorisation. 

In [11]:
# Generating a list of primes up to 1e6

# empty list for primes to be added to
primes = []

for i in range(2,1000000):

    # intitially setting is_prime to be true
    # this should be more efficient since as soon as the if statement detects a divisor it moves on
    is_prime = True

    # only need to check integers up to sqrt(i) -
    # if a factor bigger is found there must be a factor less than it too
    for j in range(2,int(sqrt(i)) + 1):

        # checking if any integer less than it is a divisor
        if i % j == 0:
            is_prime = False
            break

    # if no divisors were found, number is added to prime list
    if is_prime == True:
        primes.append(i)



In [ ]:
def prime_factorisation(n):

    # empty set for factors to be added to
    factors = []

    for p in primes:

        # checks if the current prime squared is greater than n
        if p*p > n:
            break

        # checks primes until one is divisible by n, then adds it to the list of factors
        while n % p == 0:
            factors.append(p)
            n //= p

    # if the list of primes is exhausted, add the remainder to the list of factors
    if n > 1:
        factors.append(n)

    return factors

In [ ]:
# --- More efficient method using prime factorisation and multiplicative functions ---

def aliquot_prime(x, val_limit = 1e14, iter_limit = 50):

    # creating an empty set for numbers to be added to
        aliquot_list = []
    
    
        # makes sure the function only does iterations up to the limit
        for i in range(iter_limit):

            # check to see if x goes over the value limit
            if x > val_limit:
                 print("The aliquot sequence exceeds the value limit")
                 break

            # adds the latest number to the list and resets the value of x_divisors
            aliquot_list.append(x)
            x_divisors = 1
    
            # creates a list of all the prime factors of x
            factors = prime_factorisation(x)

            # creating an empty dictionary set
            factor_counts = {}

            # goes through each prime in factors
            # if it's already an entry, adds 1 to the counter
            # if there's no entry, it creates one at 1
            for p in factors:
                if p in factor_counts:
                    factor_counts[p] += 1
                else:
                    factor_counts[p] = 1

            # using the formula for geometric series to find the sum of all positive divisors
            for p, power in factor_counts.items():
                x_divisors *= (p**(power + 1) - 1) // (p - 1)
    
            # minusing the original value to get the proper divisors and
            # replacing the old value with the new one
            x = x_divisors - x

            # stops if the sequence reaches 1 or 0
            if x == 1:
                 aliquot_list.append(1)
                 aliquot_list.append(0)
                 break
            elif x == 0:
                 aliquot_list.append(0)
                 break

            # check to see if new value is a repeat
            # if so, breaks the loop and states a loop has occured
            if x in aliquot_list:
                aliquot_list.append(x)
                print("The aliquot sequence falls into a loop starting at", x)
                break

            # check to see if the sequence goes above the iteration limit
            if i == iter_limit-1:
                 print("The aliquot sequence exceeds the iteration limit")
        
        return print(aliquot_list)